# 05  -  Deep Learning Architectures

**Goal:** Inspect the architecture of each PyTorch sequence model  -  GRU, CNN1D, LSTM, TCN, and Transformer Encoder  -  by instantiating them, printing summaries, counting parameters, and running a single forward pass.

**Modules used:** `src/models/gru_model.py`, `src/models/cnn1d_model.py`, `src/models/lstm_model.py`, `src/models/tcn_model.py`, `src/models/transformer_encoder_model.py`, `src/models/model_factory.py`

---

## 0  -  Imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models.model_factory import build_model, get_model_family, requires_sequence_input

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOOKBACK  = 21
N_FEATURES = 7
BATCH_SIZE = 16

print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

---
## 1  -  Shared config & dummy input

All torch models receive a 3-D tensor of shape `(batch, lookback, n_features)` and output raw logits of shape `(batch, 1)`.

In [ ]:
MODEL_CONFIG = {
    'gru': {
        'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'weighted_loss': True
    },
    'cnn1d': {
        'conv_channels': 64, 'kernel_size': 3, 'dropout': 0.2, 'weighted_loss': True
    },
    'lstm': {
        'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'bidirectional': False
    },
    'tcn': {
        'channels': [64, 64, 64], 'kernel_size': 3, 'dropout': 0.2
    },
    'transformer_encoder': {
        'd_model': 64, 'nhead': 4, 'num_layers': 2,
        'dim_feedforward': 128, 'dropout': 0.1, 'pooling': 'mean'
    },
}

INPUT_SHAPE = (LOOKBACK, N_FEATURES)
DUMMY_INPUT = torch.randn(BATCH_SIZE, LOOKBACK, N_FEATURES).to(DEVICE)

print(f'Dummy input shape: {DUMMY_INPUT.shape}  (batch={BATCH_SIZE}, lookback={LOOKBACK}, features={N_FEATURES})')


---
## 2  -  Helper: parameter count

In [ ]:
def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

---
## 3  -  GRU  -  Gated Recurrent Unit

The GRU processes the sequence step-by-step, maintaining a **hidden state** that serves as a compressed memory. At each time step, two gates control what to update and what to forget:
- **Update gate**  -  how much of the previous hidden state to keep.
- **Reset gate**  -  how much of the previous hidden state to ignore when computing the new candidate.

We take the **last hidden state** (at `t = lookback-1`) as the sequence representation.

In [ ]:
gru_model = build_model('gru', MODEL_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
print(gru_model)
total, trainable = count_parameters(gru_model)
print(f'\nTotal params: {total:,}  |  Trainable: {trainable:,}')

with torch.no_grad():
    out = gru_model(DUMMY_INPUT)
print(f'Output shape: {out.shape}  (raw logits)')

---
## 4  -  CNN1D  -  1-D Convolutional Neural Network

Unlike recurrent models, CNN1D applies **learnable filters** along the time axis, capturing local temporal patterns (e.g. short bursts of momentum or sudden VIX spikes).

Architecture:
```
Input (B, L, F) -> transpose -> (B, F, L)
  -> Conv1d(F -> 64, kernel=3) + ReLU + BatchNorm
  -> Conv1d(64 -> 64, kernel=3) + ReLU + BatchNorm
  -> AdaptiveAvgPool1d(1) -> (B, 64)
  -> Dropout -> Linear -> logit
```

In [ ]:
cnn_model = build_model('cnn1d', MODEL_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
print(cnn_model)
total, trainable = count_parameters(cnn_model)
print(f'\nTotal params: {total:,}  |  Trainable: {trainable:,}')

with torch.no_grad():
    out = cnn_model(DUMMY_INPUT)
print(f'Output shape: {out.shape}')

---
## 5  -  LSTM  -  Long Short-Term Memory

The LSTM extends the GRU with a separate **cell state**  -  a long-range memory lane  -  and three gates:
- **Forget gate**  -  what to erase from the cell.
- **Input gate**  -  what new information to write.
- **Output gate**  -  what part of the cell state to expose as the hidden state.

This makes it better than GRU at capturing very long-range dependencies (e.g. multi-month yield-curve regimes).

In [ ]:
lstm_model = build_model('lstm', MODEL_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
print(lstm_model)
total, trainable = count_parameters(lstm_model)
print(f'\nTotal params: {total:,}  |  Trainable: {trainable:,}')

with torch.no_grad():
    out = lstm_model(DUMMY_INPUT)
print(f'Output shape: {out.shape}')

---
## 6  -  TCN  -  Temporal Convolutional Network

TCN uses **dilated causal convolutions** stacked in residual blocks. Dilation factor doubles with each layer: 1, 2, 4, 8...

```
Layer 0 (dilation=1):  [ ] [ ] [ ]   -  receptive field: 3
Layer 1 (dilation=2):  [ ]  -  [ ]  -  [ ]   -  receptive field: 5
Layer 2 (dilation=4):  [ ]  -   -   -  [ ]  -   -   -  [ ]   -  receptive field: 9
```

Key properties:
- **Causal**  -  never looks at future time steps (mandatory for time-series forecasting).
- **Residual connections**  -  enables training deeper networks.
- **Parallel computation**  -  faster than RNNs at training time.

In [ ]:
tcn_model = build_model('tcn', MODEL_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
print(tcn_model)
total, trainable = count_parameters(tcn_model)
print(f'\nTotal params: {total:,}  |  Trainable: {trainable:,}')

with torch.no_grad():
    out = tcn_model(DUMMY_INPUT)
print(f'Output shape: {out.shape}')

---
## 7  -  Transformer Encoder

The Transformer uses **multi-head self-attention** to directly model pairwise relationships between all time steps simultaneously  -  unlike RNNs which must propagate information sequentially.

```
Input (B, L, F)
  -> Linear projection -> (B, L, d_model=64)
  -> + Learnable positional embeddings
  -> TransformerEncoderLayer x 2  (nhead=4, FFN dim=128)
  -> Mean pooling across time steps
  -> Dropout -> Linear -> logit
```

Self-attention allows the model to directly attend to e.g. the yield curve value 15 days ago when processing today's VIX reading.

In [ ]:
transformer_model = build_model('transformer_encoder', MODEL_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
print(transformer_model)
total, trainable = count_parameters(transformer_model)
print(f'\nTotal params: {total:,}  |  Trainable: {trainable:,}')

with torch.no_grad():
    out = transformer_model(DUMMY_INPUT)
print(f'Output shape: {out.shape}')

---
## 8  -  Model comparison: parameter count & architecture type

In [ ]:
model_names = ['gru', 'cnn1d', 'lstm', 'tcn', 'transformer_encoder']
models_built = [gru_model, cnn_model, lstm_model, tcn_model, transformer_model]

rows = []
for name, m in zip(model_names, models_built):
    total, trainable = count_parameters(m)
    rows.append({'model': name, 'total_params': total, 'trainable_params': trainable})

param_df = pd.DataFrame(rows)
print(param_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(param_df['model'], param_df['total_params'], color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Total Parameters')
ax.set_title('Model Size Comparison', fontsize=12)
for i, val in enumerate(param_df['total_params']):
    ax.text(val + 50, i, f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 9  -  Model factory meta-info

In [ ]:
all_models = ['logreg', 'rf', 'xgboost', 'lightgbm', 'catboost',
              'gru', 'cnn1d', 'lstm', 'tcn', 'transformer_encoder']

meta = pd.DataFrame([
    {
        'model': name,
        'family': get_model_family(name),
        'sequence_input': requires_sequence_input(name),
    }
    for name in all_models
])
meta

---
## Summary

| Model | Inductive bias | Best for |
|---|---|---|
| GRU | Sequential memory with gates | Medium-range temporal patterns |
| CNN1D | Local receptive field | Short bursts, spikes |
| LSTM | Gated cell + hidden state | Long-range regime shifts |
| TCN | Dilated causal convolutions | Hierarchical temporal patterns |
| Transformer | Global self-attention | Direct long-range dependencies |

**Next:** `06_training_deep_learning.ipynb`  -  run the full training loop with LR search, early stopping, and loss / MCC curves.